# Fundamentals 00.4 - Runtime vLLM Provider API

Objetivo: probar la ruta `vllm-runtime` de forma aislada antes de usar agentes, systems o graphs con un modelo local/GPU.

Este notebook ensena la capa provider para vLLM. vLLM no es framework: es infraestructura externa que expone una API OpenAI-compatible. Agentic Systems se conecta a esa API con `provider="vllm-runtime"`.

Regla de diseno:

```text
Agentic Systems define el contrato de ejecucion.
vllm-runtime define el backend OpenAI-compatible.
vLLM server corre fuera de la libreria, normalmente en Colab/GPU.
```


## 0) Instalacion e imports minimos

En notebooks usa `%pip`, no `%%python -m pip`.

- `%pip` instala en el kernel activo donde despues haces `import agentic_systems`.
- `%%python` ejecuta otro proceso Python; puede instalar en un entorno distinto y luego el import falla.

Para Colab/PyPI:

```python
%pip install -U pip
%pip install -U "agentic-systems[vllm]"
# o, si quieres todos los extras:
%pip install -U "agentic-systems[all]"
```

`agentic-systems[vllm]` instala el cliente OpenAI-compatible y la dependencia del servidor `vllm` cuando la version publicada del paquete ya incluye ese extra. El alias `agentic-systems[vll]` tambien existe. `agentic-systems[all]` incluye vLLM junto con los demas extras.

Si PyPI todavia entrega una version anterior de `agentic-systems`, instala `vllm` de forma explicita en el mismo kernel:

```python
%pip install -U vllm
```

Si Colab instala una rueda incompatible y ves `ImportError: libcudart.so.13`, reinstala vLLM contra CUDA 12.9 en el mismo kernel:

```python
%pip uninstall -y vllm
%pip install -U vllm --extra-index-url https://download.pytorch.org/whl/cu129
```

Esa correccion es externa a Agentic Systems: resuelve la compatibilidad binaria entre vLLM, PyTorch y CUDA del runtime T4.

Para `vllm==0.10.2`, vLLM requiere `transformers>=4.55.2` y `tokenizers>=0.21.1`. Si el server falla con `Qwen2Tokenizer has no attribute all_special_tokens_extended`, actualiza esas dependencias en el mismo kernel y reinicia runtime:

```python
%pip install -U "transformers>=4.55.2,<5" "tokenizers>=0.21.1,<0.22" "openai>=1.99.1" "huggingface_hub>=0.23.0"
```

La receta funcional descarga primero el modelo con `huggingface_hub.snapshot_download` y luego pasa la ruta local a vLLM. Esto evita diferencias entre carga remota/tokenizer/cache y hace el arranque mas reproducible en Colab/L4.

El notebook imprime la version de `vllm` con `importlib.metadata` porque `import vllm` puede fallar si la rueda CUDA es incompatible. `vllm installed: True` solo significa que el paquete existe; no prueba que sus extensiones CUDA puedan cargarse.


In [ ]:
# Ejecuta esta celda solo en Colab o en un ambiente donde el paquete no este instalado.
# En notebooks, %pip instala en el kernel activo. No uses %%python para instalar dependencias.
#
# %pip install -U pip
# %pip install -U "agentic-systems[vllm]"
# # o todos los extras:
# %pip install -U "agentic-systems[all]"
#
# Si vLLM falla con ImportError: libcudart.so.13 en Colab/T4, reinstala una rueda CUDA 12.x:
# %pip uninstall -y vllm
# %pip install -U vllm --extra-index-url https://download.pytorch.org/whl/cu129
#
# Si vLLM falla cargando Qwen con Qwen2Tokenizer/all_special_tokens_extended:
# %pip install -U "transformers>=4.55.2,<5" "tokenizers>=0.21.1,<0.22" "openai>=1.99.1" "huggingface_hub>=0.23.0"


In [ ]:
import gc
import importlib.util
from importlib.metadata import PackageNotFoundError, version as package_version
import json
import os
import subprocess
import socket
import sys
import time
from pathlib import Path
from urllib.request import Request
import urllib.request
import transformers

import agentic_systems as toolkit

print("python:", sys.executable)
print("agentic_systems:", toolkit.__name__)
print("agentic_systems version:", getattr(toolkit, "__version__", "unknown"))
print("vllm installed:", importlib.util.find_spec("vllm") is not None)
try:
    print("vllm version:", package_version("vllm"))
except PackageNotFoundError:
    print("vllm version: not installed")
print("transformers version:", transformers.__version__)


## 1) Configuracion del provider

`vllm-runtime` lee configuracion desde variables de entorno o `.env`:

| Variable | Uso |
|---|---|
| `VLLM_BASE_URL` | URL OpenAI-compatible del servidor vLLM. |
| `VLLM_MODEL` | Modelo servido por vLLM. |
| `VLLM_API_KEY` | API key para servidores compatibles; normalmente `EMPTY` local. |

El default local es `http://127.0.0.1:8000/v1` y modelo `Qwen/Qwen3-0.6B`. El notebook no guarda secretos.

| `VLLM_LOCAL_MODEL_DIR` | Ruta local opcional para cargar el modelo ya descargado. |


In [ ]:
# Valores seguros para Colab/local si todav?a no estan definidos.
os.environ.setdefault("VLLM_BASE_URL", "http://127.0.0.1:8000/v1")
os.environ.setdefault("VLLM_MODEL", "Qwen/Qwen3-0.6B")
os.environ.setdefault("VLLM_API_KEY", "EMPTY")

vllm_env = {
    "VLLM_BASE_URL": os.getenv("VLLM_BASE_URL"),
    "VLLM_MODEL": os.getenv("VLLM_MODEL"),
    "VLLM_API_KEY_configured": bool(os.getenv("VLLM_API_KEY")),
}

toolkit.show(vllm_env, title="Configuraci?n vLLM segura")


## 2) Declarar `RuntimeConfig`

`toolkit.runtime(provider="vllm-runtime")` no ejecuta el modelo. Solo declara el contrato de provider, modelo, scheduler y metadata segura.

`runtime.describe()` sirve para confirmar que leera Agentic Systems antes de hacer inferencia.


In [ ]:
scheduler = toolkit.scheduler(
    timeout_s=60,
    max_retries=0,
    max_tool_calls=4,
    max_turns=4,
    max_concurrency=1,
)

vllm_runtime = toolkit.runtime(
    provider="vllm-runtime",
    scheduler=scheduler,
)

auto_runtime = toolkit.runtime(
    provider="auto",
    scheduler=scheduler,
)

toolkit.show(vllm_runtime.describe(), title="vLLM runtime - describe")
toolkit.show(auto_runtime.describe(), title="Auto runtime - describe")


## 3) Servidor vLLM opcional en Colab/GPU

`agentic-systems` no instala ni arranca el servidor GPU `vllm`. La libreria solo habla con una API OpenAI-compatible ya levantada.

Esta celda usa la misma estrategia del template funcional de Qwen: limpia el puerto, lanza `vllm.entrypoints.openai.api_server`, fija presupuesto de VRAM, limita contexto/concurrencia para T4 y habilita parsers Qwen3 para reasoning/tool-calls.

Valores seguros para Colab T4 con `Qwen/Qwen3-0.6B`:

```text
VLLM_GPU_MEMORY_UTILIZATION=0.40
VLLM_MAX_MODEL_LEN=2048
VLLM_MAX_NUM_SEQS=4
VLLM_TOOL_CALL_PARSER=auto  # hermes para Qwen no-Coder; qwen3_xml para Qwen3-Coder
VLLM_REASONING_PARSER=qwen3
```

Si tu GPU tiene mas margen, puedes subir esos valores, pero primero valida `/v1/models`. Si falla, revisa `vllm_language.log`.


In [ ]:
from dataclasses import dataclass
from typing import Optional


@dataclass
class LocalModelPaths:
    llm_dir: str
    emb_dir: str | None = None
    rerank_dir: str | None = None


@dataclass
class VllmConfig:
    host: str = "127.0.0.1"
    llm_port: int = 8000
    emb_port: int = 8001
    rerank_port: int = 8002
    llm_gpu_util: float = 0.40
    emb_gpu_util: float = 0.30
    rerank_gpu_util: float = 0.30
    llm_max_len: int = 2048
    emb_max_len: int = 512
    rerank_max_len: int = 512
    llm_max_num_seqs: int = 4
    emb_max_num_seqs: int = 4
    rerank_max_num_seqs: int = 4
    llm_served_name: str | None = None
    emb_served_name: str = "agentic-embed"
    rerank_served_name: str = "agentic-rerank"
    tool_call_parser: str = "hermes"
    enable_reasoning: bool = True
    reasoning_parser: Optional[str] = "qwen3"
    start_llm_server: bool = True
    start_emb_server: bool = False
    start_rerank_server: bool = False


@dataclass
class VllmEndpoints:
    llm_base_url: str | None = None
    emb_base_url: str | None = None
    rerank_base_url: str | None = None


@dataclass
class VllmServers:
    llm_proc: subprocess.Popen | None = None
    emb_proc: subprocess.Popen | None = None
    rerank_proc: subprocess.Popen | None = None
    llm_log_path: str | None = None
    emb_log_path: str | None = None
    rerank_log_path: str | None = None


def ensure_model_downloaded(model_id: str, local_dir: str | Path) -> str:
    from huggingface_hub import snapshot_download

    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    return snapshot_download(repo_id=model_id, local_dir=str(local_dir), local_dir_use_symlinks=False)


def prepare_local_models(*, llm_model_id: str, emb_model_id: str | None = None, rerank_model_id: str | None = None, base_dir: str = "LM_MODEL") -> LocalModelPaths:
    base = Path(base_dir)
    llm_dir = ensure_model_downloaded(llm_model_id, base / "LLM_MAIN")
    emb_dir = ensure_model_downloaded(emb_model_id, base / "Embedding") if emb_model_id else None
    rerank_dir = ensure_model_downloaded(rerank_model_id, base / "Reranker") if rerank_model_id else None
    return LocalModelPaths(llm_dir=llm_dir, emb_dir=emb_dir, rerank_dir=rerank_dir)


def cleanup_gpu_and_vllm_processes() -> None:
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    os.system('pkill -f "vllm.entrypoints.openai.api_server" || true')
    os.system('pkill -f "vllm" || true')


def free_port_if_needed(host: str, port: int) -> None:
    try:
        sock = socket.socket()
        sock.settimeout(0.5)
        is_open = sock.connect_ex((host, port)) == 0
        sock.close()
        if is_open:
            os.system(f"fuser -k {port}/tcp || true")
    except Exception:
        pass


def url_open_no_proxy(url: str, *, timeout: float = 2.5) -> dict:
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    urllib.request.install_opener(opener)
    request = Request(url, headers={"Authorization": f"Bearer {os.getenv('VLLM_API_KEY', 'EMPTY')}"})
    with opener.open(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def tail_text(path: str, *, max_chars: int = 6000) -> str:
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as handle:
            return handle.read()[-max_chars:]
    except FileNotFoundError:
        return ""


def wait_until_ready(url: str, process: subprocess.Popen, log_path: str, *, seconds: int = 240, sleep_s: float = 2.0) -> bool:
    start = time.time()
    while time.time() - start < seconds:
        if process.poll() is not None:
            return False
        try:
            url_open_no_proxy(url, timeout=2.5)
            return True
        except Exception:
            time.sleep(sleep_s)
    return False


def launch_vllm_server(name: str, model_dir: str, host: str, port: int, served_model_name: str | None, gpu_util: float, max_model_len: int | None, max_num_seqs: int | None, extra_flags: list[str] | None = None) -> tuple[subprocess.Popen, str, list[str]]:
    free_port_if_needed(host, port)
    log_path = f"vllm_{name}.log"
    cmd = [
        sys.executable,
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        model_dir,
        "--host",
        host,
        "--port",
        str(port),
        "--gpu-memory-utilization",
        str(gpu_util),
    ]
    if max_model_len is not None:
        cmd += ["--max-model-len", str(max_model_len)]
    if max_num_seqs is not None:
        cmd += ["--max-num-seqs", str(max_num_seqs)]
    if served_model_name:
        cmd += ["--served-model-name", served_model_name]
    if extra_flags:
        cmd += list(extra_flags)
    process = subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
    return process, log_path, cmd


def start_local_vllm_servers(model_paths: LocalModelPaths, config: VllmConfig, *, set_env: bool = True) -> tuple[VllmEndpoints, VllmServers, dict]:
    llm_proc = None
    llm_log = None
    cmd = None
    if config.start_llm_server:
        extra_flags = ["--enable-auto-tool-choice", "--tool-call-parser", config.tool_call_parser]
        if config.enable_reasoning and config.reasoning_parser:
            extra_flags += ["--reasoning-parser", config.reasoning_parser]
        llm_proc, llm_log, cmd = launch_vllm_server(
            "language",
            model_paths.llm_dir,
            config.host,
            config.llm_port,
            config.llm_served_name,
            config.llm_gpu_util,
            config.llm_max_len,
            config.llm_max_num_seqs,
            extra_flags,
        )
    llm_base = f"http://{config.host}:{config.llm_port}/v1" if config.start_llm_server else None
    ok_llm = True
    if llm_proc is not None and llm_log is not None and llm_base is not None:
        ok_llm = wait_until_ready(f"{llm_base}/models", llm_proc, llm_log)
    if set_env and ok_llm and llm_base:
        os.environ["VLLM_BASE_URL"] = llm_base
        os.environ["VLLM_MODEL"] = config.llm_served_name or os.getenv("VLLM_MODEL", "Qwen/Qwen3-0.6B")
        os.environ.setdefault("VLLM_API_KEY", "EMPTY")
    endpoints = VllmEndpoints(llm_base_url=llm_base)
    servers = VllmServers(llm_proc=llm_proc, llm_log_path=llm_log)
    meta = {"ok_llm": ok_llm, "cmd": cmd, "log_tail": tail_text(llm_log or "")}
    return endpoints, servers, meta


def vllm_environment_blockers() -> list[str]:
    blockers = []
    try:
        vllm_version = package_version("vllm")
    except PackageNotFoundError:
        blockers.append("vllm no esta instalado")
        vllm_version = "not installed"
    transformers_major = int(transformers.__version__.split(".", 1)[0])
    if transformers_major >= 5:
        blockers.append(f"transformers {transformers.__version__} es incompatible con vllm 0.10.x; usa transformers>=4.55.2,<5")
    if vllm_version != "0.10.2":
        blockers.append(f"vllm {vllm_version} no es la version validada para Colab CUDA 12.8; usa vllm==0.10.2")
    return blockers


# Misma receta operacional que el template Qwen funcional.
START_LOCAL_VLLM_SERVER = False
LLM_MODEL_ID = os.getenv("VLLM_MODEL", "Qwen/Qwen3-0.6B")
VLLM_MODE = os.getenv("VLLM_MODE", "FAST")  # FAST, MEDIUM, POWER

if START_LOCAL_VLLM_SERVER:
    blockers = vllm_environment_blockers()
    if blockers:
        server_status = {
            "status": "blocked",
            "reason": "Entorno vLLM incompatible",
            "blockers": blockers,
            "fix": "%pip uninstall -y vllm transformers tokenizers && %pip install -U vllm==0.10.2 'transformers>=4.55.2,<5' 'tokenizers>=0.21.1,<0.22' openai>=1.99.1 huggingface_hub>=0.23.0; reinicia runtime",
        }
    else:
        cleanup_gpu_and_vllm_processes()
        model_paths = prepare_local_models(llm_model_id=LLM_MODEL_ID, base_dir="LM_MODEL")
        if VLLM_MODE == "POWER":
            cfg = VllmConfig(llm_gpu_util=0.90, llm_max_len=32768, llm_max_num_seqs=1, llm_served_name=LLM_MODEL_ID, tool_call_parser="hermes", reasoning_parser="qwen3")
        elif VLLM_MODE == "MEDIUM":
            cfg = VllmConfig(llm_gpu_util=0.55, llm_max_len=4096, llm_max_num_seqs=6, llm_served_name=LLM_MODEL_ID, tool_call_parser="hermes", reasoning_parser="qwen3")
        else:
            cfg = VllmConfig(llm_gpu_util=0.40, llm_max_len=2048, llm_max_num_seqs=4, llm_served_name=LLM_MODEL_ID, tool_call_parser="hermes", reasoning_parser="qwen3")
        endpoints, servers, meta = start_local_vllm_servers(model_paths, cfg, set_env=True)
        server_status = {
            "status": "started" if meta["ok_llm"] else "starting_or_failed",
            "pid": servers.llm_proc.pid if servers.llm_proc else None,
            "returncode": servers.llm_proc.poll() if servers.llm_proc else None,
            "model_path": model_paths.llm_dir,
            "base_url": endpoints.llm_base_url,
            "log_path": servers.llm_log_path,
            "cmd": meta["cmd"],
            "log_tail": meta["log_tail"],
        }
else:
    server_status = {"status": "skipped", "reason": "START_LOCAL_VLLM_SERVER=False"}

toolkit.show(server_status, title="vLLM server launch opcional")


## 4) Health check opcional del servidor

Esta celda intenta consultar `/models` en el servidor vLLM OpenAI-compatible. Si no hay servidor levantado, no falla el notebook: reporta `skipped`.


In [ ]:
models_url = os.getenv("VLLM_BASE_URL", "http://127.0.0.1:8000/v1").rstrip("/") + "/models"

try:
    payload = url_open_no_proxy(models_url, timeout=3)
    toolkit.show({"status": "ok", "models_url": models_url, "response": payload}, title="vLLM server health")
    VLLM_SERVER_AVAILABLE = True
except Exception as exc:
    toolkit.show({"status": "skipped", "models_url": models_url, "reason": str(exc), "log_tail": tail_text("vllm_language.log")}, title="vLLM server health")
    VLLM_SERVER_AVAILABLE = False


## 5) Tool smoke opcional con `vllm-runtime`

Este smoke usa una tool normal de Agentic Systems. Si el servidor vLLM esta disponible y soporta tool calling compatible, el agente puede llamarla.

Si el servidor no esta levantado, la celda reporta `skipped` para que el notebook siga siendo portable en local, VSCode y Colab.


In [ ]:
@toolkit.tool
def sumar(a: int, b: int) -> dict:
    """Suma dos enteros."""
    return {"result": a + b}

policy = toolkit.RunPolicy(
    max_turns=4,
    max_tool_calls=2,
    temperature=0.0,
    tool_choice="auto",
    repair=True,
    max_repairs=1,
    trace="compact",
    strict=True,
)

if not VLLM_SERVER_AVAILABLE:
    toolkit.show({"status": "skipped", "reason": "Servidor vLLM no disponible."}, title="Tool smoke vLLM")
    result = None
else:
    system = toolkit.AgenticSystem(runtime=vllm_runtime)
    agent = system.agent(
        name="qwen_calculator",
        instructions="Usa la tool sumar para resolver la petici?n y responde breve.",
        tools=[sumar],
        engine="vllm-runtime",
        runtime=vllm_runtime,
        policy=policy,
    )
    result = agent.run("Suma 10 y 20 usando la tool sumar.")
    toolkit.human_result(result)


## 6) Cierre de API

Este notebook cubre la ruta provider vLLM sin introducir frameworks externos. La composicion con `AgenticSystem`, LangGraph, Strands u OpenAI Agents se ensena en notebooks posteriores.


In [ ]:
api_coverage = [
    {"api": "toolkit.runtime(provider='vllm-runtime')", "description": "Declara vLLM como provider canonico."},
    {"api": "toolkit.runtime(provider='auto')", "description": "Selecciona vLLM automaticamente cuando VLLM_BASE_URL esta configurado."},
    {"api": "RuntimeConfig.describe", "description": "Muestra resolucion y configuracion segura."},
    {"api": "toolkit.scheduler", "description": "Declara limites de ejecucion."},
    {"api": "toolkit.RunPolicy", "description": "Controla loops, tool calls y trazas."},
    {"api": "toolkit.human_result", "description": "Renderiza resultados reales de ejecucion."},
    "VLLM_RUNTIME_ENGINE",
]

toolkit.show({"notebook": "00_runtime_vllm_provider_api.ipynb", "api_coverage": api_coverage}, title="API coverage")
